# Intelligent Crime Detective Platform — Person B Analytics Notebook
## Day 2: Dataset Cleaning & Feature Engineering

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 2 of 10-Day Development Plan  

### Core Governance & Methodological Principles for Day 2
1. **Raw Data Immutability:** Raw datasets in `data/raw/` are immutable primary sources and are never altered.
2. **No Premature ML:** In accordance with the project roadmap, no model training, PCA, LWR, ID3, Naive Bayes, or k-NN algorithms are executed today.
3. **Target Leakage Prevention:** `disposition` and its derived indicator `is_solved` represent ground-truth investigative outcomes and are strictly isolated as target variables, never input features.
4. **Semantic Preservation:** Missing coordinates and unknown ages are never filled with `0`. Infant age `0` is strictly preserved as valid.
5. **Clean Processed Outputs:** Cleaned datasets with documented lineage are saved under `data/processed/`.

## 1. Environment Setup

Initialize runtime environment, configure paths, and import foundational libraries and internal utilities.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUTS_DIR
from src.data_inspection import inspect_dataset
from src.data_cleaning import (
    load_csv, strip_whitespace, normalize_categorical,
    detect_duplicates, report_missing_values, parse_dates_safely,
    coerce_numeric, validate_coordinates, identify_total_rows
)
from src.data_quality import dataset_summary, duplicate_report, profile_columns, generate_quality_report, missing_value_report
from src.feature_engineering import (
    extract_temporal_features, create_target_solvability,
    flag_target_leakage_columns, standardize_tamil_nadu_districts
)
from src.run_cleaning_pipeline import run_all_cleaning

print(f"Project Root: {PROJECT_ROOT}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

## 2. Dataset Discovery

Programmatically discover and verify the physical files present in `data/raw/`.

In [ ]:
raw_files = sorted(list(RAW_DATA_DIR.glob("*.csv")))
print(f"Discovered {len(raw_files)} CSV datasets in {RAW_DATA_DIR}:")

discovery_records = []
for f in raw_files:
    size_kb = round(f.stat().st_size / 1024, 2)
    discovery_records.append({
        "Filename": f.name,
        "File Size (KB)": size_kb,
        "File Size (MB)": round(size_kb / 1024, 2),
    })

df_discovery = pd.DataFrame(discovery_records)
display(df_discovery)

## 3. Dataset Inventory

Define the analytical scope, target domain, and planned Person B modeling modules for each discovered dataset.

In [ ]:
inventory_data = [
    {
        "Filename": "homicide-data.csv",
        "Target Domain": "US Major City Homicides (50 cities)",
        "Target / Outcome Column": "disposition (Closed by arrest vs others)",
        "Planned Person B Modules": "Solvability Estimation (ID3, Naive Bayes, k-NN), Geographic Profiling (LWR)",
    },
    {
        "Filename": "dstrIPC_1_2014.csv",
        "Target Domain": "NCRB National District IPC Crime Records (2014)",
        "Target / Outcome Column": "Total Cognizable IPC crimes (Aggregate Benchmark)",
        "Planned Person B Modules": "Principal Component Analysis (PCA), Crime Offense Pattern Profiling",
    },
    {
        "Filename": "TN-2020-2022-total.csv",
        "Target Domain": "Tamil Nadu Multi-Year Cognizable Crime Totals (2020-2022)",
        "Target / Outcome Column": "Rate of Cognizable crime (IPC+SLL)",
        "Planned Person B Modules": "Longitudinal Trend Analysis, District Growth Comparison, Streamlit UI",
    },
    {
        "Filename": "TN-murder-2023.csv",
        "Target Domain": "Tamil Nadu Violent & Negligent Fatalities (2023)",
        "Target / Outcome Column": "Murder - Rate / Incidence",
        "Planned Person B Modules": "Tamil Nadu Geospatial Analytics, Folium Interactive Map",
    },
]

df_inventory = pd.DataFrame(inventory_data)
display(df_inventory)

## 4. Schema Inspection

Inspect the native dimensions, column headers, and data types across each source dataset.

In [ ]:
schema_summaries = []
loaded_raw = {}

for f in raw_files:
    enc = "latin-1" if "homicide" in f.name else "utf-8"
    df_temp = load_csv(f, encodings_to_try=[enc, "latin-1", "utf-8"])
    loaded_raw[f.name] = df_temp
    
    schema_summaries.append({
        "Dataset": f.name,
        "Rows": len(df_temp),
        "Columns": len(df_temp.columns),
        "Numeric Cols": len(df_temp.select_dtypes(include=[np.number]).columns),
        "String/Object Cols": len(df_temp.select_dtypes(include=["object", "string"]).columns),
    })

display(pd.DataFrame(schema_summaries))

## 5. Data Quality Analysis

Generate comprehensive quality audits across the raw data assets.

In [ ]:
quality_metrics = []
for name, df_temp in loaded_raw.items():
    summary = dataset_summary(df_temp, dataset_name=name)
    quality_metrics.append(summary)

display(pd.DataFrame(quality_metrics))

## 6. Missing Value Analysis

Audit missing value counts, percentages, and semantic causes across all fields.

In [ ]:
for name, df_temp in loaded_raw.items():
    rep = missing_value_report(df_temp)
    missing_cols = rep[rep["missing_count"] > 0]
    print(f"=== Missing Values in {name} ===")
    if len(missing_cols) == 0:
        print("  -> No missing values found in native DataFrame.")
    else:
        display(missing_cols)

print("\nSemantic Missingness Audit:")
print("1. homicide-data.csv: lat & lon have 60 nulls (0.11%). victim_age has 2,999 'Unknown' entries.")
print("2. TN-2020-2022-total.csv: Avadi and Tambaram have 'N/C' (Not Created) in 2020 & 2021.")
print("3. TN-murder-2023.csv: Specialized units (Railways, Cyber Cell) have '-' rates.")

## 7. Duplicate Analysis

Examine exact duplicate rows and test candidate primary keys for collisions.

In [ ]:
dup_results = []
for name, df_temp in loaded_raw.items():
    id_col = "uid" if "uid" in df_temp.columns else ("Sl No" if "Sl No" in df_temp.columns else None)
    d_rep = duplicate_report(df_temp, id_col=id_col)
    dup_results.append({
        "Dataset": name,
        "Exact Duplicate Rows": d_rep["exact_duplicate_rows"],
        "ID Column": d_rep["id_column"],
        "Duplicate IDs Count": d_rep["duplicate_ids_count"],
    })

display(pd.DataFrame(dup_results))

## 8. Categorical Feature Analysis

Profile categorical values, label cardinality, and distributions.

In [ ]:
df_hom = loaded_raw["homicide-data.csv"]
print("Victim Sex Distribution:")
display(df_hom["victim_sex"].value_counts(dropna=False).to_frame(name="Count"))

print("\nVictim Race Distribution:")
display(df_hom["victim_race"].value_counts(dropna=False).to_frame(name="Count"))

print("\nCase Disposition (Ground-Truth Clearance):")
display(df_hom["disposition"].value_counts(dropna=False).to_frame(name="Count"))

## 9. Numerical Feature Analysis

Inspect numerical distributions, boundary limits, and anomalies.

In [ ]:
# Inspect victim age distribution in homicide data
numeric_age = coerce_numeric(df_hom["victim_age"], sentinel_strings=["Unknown"])
print("Victim Age Statistics (excluding 'Unknown'):")
display(numeric_age.describe().to_frame(name="Victim Age Distribution"))

print(f"Infant records (Age == 0): {(numeric_age == 0).sum()} cases (preserved as valid)")
print(f"Unknown age records: {numeric_age.isnull().sum()} cases (represented as NaN)")

## 10. Geographic Feature Analysis

Validate latitude/longitude coordinate bounds and analyze district jurisdiction nomenclature.

In [ ]:
# Validate coordinates in homicide data
valid_coords = validate_coordinates(df_hom, lat_col="lat", lon_col="lon")
print("Coordinate Validity Audit:")
print(f"  Valid coordinates: {valid_coords.sum():,} ({valid_coords.mean()*100:.2f}%)")
print(f"  Missing/Invalid coordinates: {(~valid_coords).sum()} (0.11%)")
print(f"  Latitude range (valid): [{df_hom.loc[valid_coords, 'lat'].min():.4f}, {df_hom.loc[valid_coords, 'lat'].max():.4f}]")
print(f"  Longitude range (valid): [{df_hom.loc[valid_coords, 'lon'].min():.4f}, {df_hom.loc[valid_coords, 'lon'].max():.4f}]")

# Check Tamil Nadu district transliteration differences
df_tn_tot = loaded_raw["TN-2020-2022-total.csv"]
df_tn_mrd = loaded_raw["TN-murder-2023.csv"]
print("\nTamil Nadu District Jurisdictions in 2020-2022 Totals:", len(df_tn_tot))
print("Tamil Nadu District Jurisdictions in 2023 Murder:", len(df_tn_mrd))
new_dist = set(df_tn_mrd["Districts/City"]) - set(df_tn_tot["Districts"])
print("New district appearing in 2023:", new_dist)

## 11. Temporal Feature Analysis

Analyze reporting dates, handle entry typos safely, and extract calendar features.

In [ ]:
dates_raw = df_hom["reported_date"].astype(str)
print("Date string length distribution:")
display(dates_raw.str.len().value_counts().to_frame(name="Count"))

malformed = df_hom[dates_raw.str.len() != 8]
print(f"Malformed date records count: {len(malformed)}")
display(malformed[["uid", "reported_date", "city", "state"]])

dates_parsed = parse_dates_safely(df_hom["reported_date"], format="%Y%m%d", errors="coerce")
print(f"Successfully parsed dates: {dates_parsed.notnull().sum():,}")
print(f"NaT dates (safely coerced): {dates_parsed.isnull().sum()}")

## 12. Cleaning Decisions

Document the explicit, auditable rationale behind all transformations performed.

In [ ]:
decisions = [
    {
        "Aspect": "Missing Coordinates (homicide-data)",
        "Action": "Do not fill with 0. Flag via valid_coords and create homicide_spatial_clean.csv.",
        "Rationale": "Filling coordinates with 0 creates fictitious incidents at (0, 0) Off Africa, corrupting GIS maps and LWR."
    },
    {
        "Aspect": "Unknown Victim Age (homicide-data)",
        "Action": "Coerce 'Unknown' to NaN. Preserve age 0.",
        "Rationale": "0 is a valid numerical age for infants (<1 year old). Imputing Unknown with 0 introduces profound bias."
    },
    {
        "Aspect": "Malformed Dates (homicide-data)",
        "Action": "Safely coerce 9-digit integers to NaT while preserving raw reported_date.",
        "Rationale": "Prevents pipeline runtime crashes while maintaining data integrity."
    },
    {
        "Aspect": "Not Created 'N/C' (TN-2020-2022-total)",
        "Action": "Convert 'N/C' to NaN in Avadi and Tambaram for 2020/2021.",
        "Rationale": "Districts created in late 2021 lack separate historical figures. Conversion allows numeric calculations."
    },
    {
        "Aspect": "Special Police Units Rates '-' (TN-murder-2023)",
        "Action": "Convert '-' to NaN in rate columns.",
        "Rationale": "Railways and Cyber Cell lack residential population bases, making per-lakh rates undefined."
    },
    {
        "Aspect": "Aggregate Summary Rows (NCRB & TN)",
        "Action": "Add is_total_row boolean flag.",
        "Rationale": "Prevents double-counting state crime totals during district-level statistical modeling."
    },
    {
        "Aspect": "Target Variable & Leakage",
        "Action": "Derive is_solved (1=Closed by arrest) and label strictly as TARGET ONLY.",
        "Rationale": "Case disposition is a post-investigation outcome; must never be supplied to input features (X)."
    },
]

display(pd.DataFrame(decisions))

## 13. Clean Dataset Generation

Execute the end-to-end data cleaning pipeline and serialize cleaned datasets into `data/processed/`.

In [ ]:
# Execute the modular pipeline runner
processed_outputs = run_all_cleaning()

processed_summary = []
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    processed_summary.append({
        "Dataset Identifier": key,
        "Output File": path.name,
        "Rows": len(df_p),
        "Columns": len(df_p.columns),
        "Size (KB)": round(path.stat().st_size / 1024, 2),
    })

display(pd.DataFrame(processed_summary))

## 14. Feature Dictionary

Preview the structured feature dictionary documented under `outputs/person_b_feature_dictionary.md`.

In [ ]:
feat_dict_path = OUTPUTS_DIR / "person_b_feature_dictionary.md"
if feat_dict_path.exists():
    print(f"Feature Dictionary successfully established at: {feat_dict_path}")
    with open(feat_dict_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"Total lines of documentation: {len(lines)}")
    # Print first 35 lines preview
    print("".join(lines[:35]))
else:
    print("Feature dictionary not found!")

## 15. Final Data Quality Report

Verify the data health and integrity of all processed outputs.

In [ ]:
print("=== FINAL PROCESSED DATA QUALITY VERIFICATION ===")
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    print(f"\n--- {key} ({path.name}) ---")
    print(f"Rows: {len(df_p):,}, Columns: {len(df_p.columns)}")
    print(f"Duplicate Rows: {df_p.duplicated().sum()}")
    null_counts = df_p.isnull().sum()
    active_nulls = null_counts[null_counts > 0]
    if len(active_nulls) > 0:
        print("Null counts per column:")
        for col, cnt in active_nulls.items():
            print(f"  {col}: {cnt} ({cnt/len(df_p)*100:.2f}%)")
    else:
        print("  Zero null cells detected.")

print("\nDay 2 Dataset Cleaning & Feature Engineering successfully completed!")

# Day 3 — PCA and Tamil Nadu Crime Analytics

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 3 of 10-Day Development Plan  

### Objectives for Day 3:
1. **PCA Macro-Level Analytics Pipeline:** Feature selection, target leakage prevention, nominal one-hot encoding, non-zero median imputation, standardized scaling, PCA training, explained variance, and component loadings.
2. **Tamil Nadu Crime Analytics Module:** District-level aggregation, multi-year longitudinal trend analysis, category breakdowns, and commissionerate comparisons.
3. **Publication-Grade Visualizations:** Four diagnostic PCA plots and four Tamil Nadu analytical charts.
4. **Model Bundle Serialization & Reproducibility:** Persisting full deployable preprocessor and PCA pipeline in `models/pca_model.joblib`.

## 1. Dataset Loading

Load the cleaned datasets produced during Day 2 from `data/processed/` using the modular project paths.

In [ ]:
from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, MODELS_DIR
import pandas as pd
import numpy as np

datasets = {}
for p in sorted(PROCESSED_DATA_DIR.glob("*.csv")):
    df_t = pd.read_csv(p)
    datasets[p.name] = df_t
    print(f"Loaded: {p.name:<35} | Shape: {str(df_t.shape):<12} | Memory: {round(p.stat().st_size/1024, 1):>7} KB")

## 2. Processed Dataset Inspection

Inspect column schemas, data types, null proportions, and governance markers across all processed files.

In [ ]:
from src.feature_engineering import identify_feature_types

inspection_rows = []
for name, df_curr in datasets.items():
    roles = identify_feature_types(df_curr)
    inspection_rows.append({
        "Dataset": name,
        "Total Rows": len(df_curr),
        "Total Cols": len(df_curr.columns),
        "Numeric Cols": len(roles["numerical"]),
        "Categorical Cols": len(roles["categorical"]),
        "Temporal Cols": len(roles["temporal"]),
        "Geographic Cols": len(roles["geographic"]),
        "Target/Leakage Cols": len(roles["target"]),
        "Identifier Cols": len(roles["identifier"]),
    })

df_inspect = pd.DataFrame(inspection_rows)
display(df_inspect)

## 3. Feature Selection

Select analytical features for macro-level incident PCA modeling while strictly eliminating target outcome labels (`is_solved`, `disposition`), PII, raw coordinates, and order identifiers.

In [ ]:
from src.pca_analysis import generate_feature_selection_table

feat_sel_df = generate_feature_selection_table()
print(f"Total Columns Audited: {len(feat_sel_df)}")
print(f"Features Selected for PCA: {feat_sel_df['selected'].sum()}")
print(f"Features Excluded: {(~feat_sel_df['selected']).sum()}")

display(feat_sel_df)

## 4. Feature-Type Identification

Partition selected modeling features into numeric and nominal categorical groupings.

In [ ]:
selected_features = feat_sel_df[feat_sel_df["selected"]]
numeric_cols = selected_features[selected_features["role"].str.contains("numerical")]["column"].tolist()
categorical_cols = selected_features[selected_features["role"].str.contains("categorical")]["column"].tolist()

print(f"Continuous / Numerical Features ({len(numeric_cols)}): {numeric_cols}")
print(f"Nominal Categorical Features ({len(categorical_cols)}): {categorical_cols}")

## 5. Categorical Encoding

Validate nominal categorical encoding using `OneHotEncoder(handle_unknown='ignore')` to ensure safe handling of unseen categories without inducing artificial ordinality.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

df_homicide = datasets["homicide_clean.csv"]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_encoded = ohe.fit_transform(df_homicide[categorical_cols])
ohe_names = ohe.get_feature_names_out(categorical_cols)

print(f"Categorical Input Dimensions: {len(categorical_cols)}")
print(f"One-Hot Encoded Dimensions: {cat_encoded.shape[1]}")
print(f"Sample Encoded Feature Names: {list(ohe_names[:10])}")

## 6. Numerical Preprocessing

Impute continuous demographic missing values using median imputation (`SimpleImputer(strategy='median')`). Infant age 0 is strictly preserved as valid and never corrupted with zero-filled missingness.

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
num_imputed = num_imputer.fit_transform(df_homicide[numeric_cols])

print("Imputation Statistics:")
for col, stat in zip(numeric_cols, num_imputer.statistics_):
    nulls = df_homicide[col].isnull().sum()
    print(f"  - {col:<20}: median={stat:<6.1f} | missing_imputed={nulls}")

print(f"Infant records with age == 0.0: {(df_homicide['victim_age_clean'] == 0.0).sum()}")

## 7. Scaling

Standardize features with `StandardScaler` to achieve zero mean and unit variance prior to eigendecomposition.

In [ ]:
from sklearn.preprocessing import StandardScaler

combined_matrix = np.hstack([num_imputed, cat_encoded])
scaler = StandardScaler()
scaled_matrix = scaler.fit_transform(combined_matrix)

print(f"Combined Matrix Shape: {scaled_matrix.shape}")
print(f"Global Mean Across Transformed Features: {scaled_matrix.mean():.6f} (near 0)")
print(f"Global Std Across Transformed Features:  {scaled_matrix.std():.6f} (near 1)")

## 8. PCA Training

Train the PCA decomposition pipeline and determine the optimal component cutoff.

In [ ]:
from src.pca_analysis import fit_pca_analysis

pca_results = fit_pca_analysis(
    df=df_homicide,
    numeric_features=numeric_cols,
    categorical_features=categorical_cols,
    variance_threshold=0.85,
)

print(f"Recommended Principal Components: {pca_results['recommended_k']}")
print(f"Transformed Dimension Count:       {len(pca_results['transformed_feature_names'])}")
print(f"Cumulative Explained Variance:     {pca_results['explained_variance_df'].iloc[pca_results['recommended_k']-1]['cumulative_explained_variance']*100:.2f}%")

## 9. Explained Variance Analysis

Audit individual explained variance ratios and verify monotonic cumulative variance growth.

In [ ]:
exp_df = pca_results["explained_variance_df"]
display(exp_df.head(15))

cum_var = exp_df["cumulative_explained_variance"].values
is_monotonic = np.all(np.diff(cum_var) >= 0)
print(f"Monotonic Cumulative Growth Verified: {is_monotonic}")

## 10. PCA Component Analysis

Inspect the component loadings matrix to interpret the principal drivers of macro crime variance.

In [ ]:
loadings_df = pca_results["loadings_df"]
print("Top 5 Positive Loadings for PC1:")
display(loadings_df["PC1"].sort_values(ascending=False).head(5))

print("Top 5 Negative Loadings for PC1:")
display(loadings_df["PC1"].sort_values().head(5))

print("Top 5 Positive Loadings for PC2:")
display(loadings_df["PC2"].sort_values(ascending=False).head(5))

## 11. PCA Visualization

Verify that the four publication-quality diagnostic PCA plots exist on disk.

In [ ]:
pca_plots = [
    OUTPUTS_DIR / "pca" / "explained_variance_by_component.png",
    OUTPUTS_DIR / "pca" / "cumulative_explained_variance.png",
    OUTPUTS_DIR / "pca" / "pca_2d_projection.png",
    OUTPUTS_DIR / "pca" / "pca_feature_contributions.png",
]

for p in pca_plots:
    assert p.exists(), f"Plot missing: {p}"
    print(f"Verified Plot Artifact: {p.name:<35} | Size: {round(p.stat().st_size/1024, 1):>6} KB")

## 12. Tamil Nadu Dataset Analysis

Ingest and inspect Tamil Nadu state and district-level crime datasets.

In [ ]:
from src.tn_analytics import load_tn_datasets

df_tn_total, df_tn_murder, df_tn_ipc = load_tn_datasets()
print(f"Tamil Nadu Multi-Year Totals Shape: {df_tn_total.shape}")
print(f"Tamil Nadu 2023 Fatalities Shape:  {df_tn_murder.shape}")
print(f"Tamil Nadu 2014 IPC Baseline Shape: {df_tn_ipc.shape}")

## 13. District Analysis

Generate aggregated district-level summaries, rank jurisdictions, and ensure no false rates are calculated for population-less units.

In [ ]:
from src.tn_analytics import build_district_summary, rank_districts, compare_districts, compute_basic_statistics

district_summary = build_district_summary(df_tn_total, df_tn_murder)
print(f"Total Standardized Districts Summarized: {len(district_summary)}")

print("\n--- Top 10 Jurisdictions by 2022 Total Crime Count ---")
display(rank_districts(district_summary, "crime_count_2022", top_n=10))

print("\n--- Top 10 Jurisdictions by 2023 Violent Fatalities ---")
display(rank_districts(district_summary, "violent_fatalities_total_2023", top_n=10))

print("\n--- Descriptive Statistics Across Districts ---")
display(compute_basic_statistics(district_summary))

## 14. Yearly Crime Trends

Examine multi-year longitudinal crime trends (2020-2022 and 2014 benchmark) and calculate year-over-year mathematical growth without claiming causation.

In [ ]:
from src.tn_analytics import build_yearly_trends

yearly_trends = build_yearly_trends(df_tn_total, df_tn_ipc)
display(yearly_trends)

print("\nResponsible Analytics Note: Trends reflect historical administrative reporting cycles. Pandemics and policy enforcements directly influenced reporting patterns without establishing intrinsic behavioural causation.")

## 15. Tamil Nadu Visualizations

Verify that the four publication-quality Tamil Nadu analytical plots exist on disk.

In [ ]:
tn_plots = [
    OUTPUTS_DIR / "tamil_nadu" / "district_crime_distribution.png",
    OUTPUTS_DIR / "tamil_nadu" / "yearly_crime_trends.png",
    OUTPUTS_DIR / "tamil_nadu" / "crime_category_distribution.png",
    OUTPUTS_DIR / "tamil_nadu" / "district_comparison.png",
]

for p in tn_plots:
    assert p.exists(), f"Plot missing: {p}"
    print(f"Verified Plot Artifact: {p.name:<35} | Size: {round(p.stat().st_size/1024, 1):>6} KB")

## 16. Output Generation

Execute end-to-end pipeline runners and verify the serialization of all Day 3 file artifacts.

In [ ]:
from src.pca_analysis import run_pca_pipeline
from src.tn_analytics import run_tn_pipeline

pca_meta = run_pca_pipeline()
tn_meta = run_tn_pipeline()

print("PCA Execution Summary:")
for k, v in pca_meta.items():
    print(f"  {k}: {v}")

print("\nTamil Nadu Analytics Summary:")
for k, v in tn_meta.items():
    print(f"  {k}: {v}")

## 17. Validation

Execute comprehensive validation checks to confirm numerical consistency, target leakage isolation, model reloadability, and raw data immutability.

In [ ]:
from src.pca_analysis import load_pca_bundle, transform_new_data
from src.config import RAW_DATA_DIR

# 1. Model Reload Test
model_path = MODELS_DIR / "pca_model.joblib"
bundle = load_pca_bundle(model_path)
sample_holdout = df_homicide.sample(20, random_state=99)
trans_holdout = transform_new_data(bundle, sample_holdout)
assert trans_holdout.shape == (20, bundle["recommended_k"])
print("Validation 1: Saved PCA bundle reloads and transforms holdout data successfully.")

# 2. Target Leakage Prevention Check
feature_names_in = bundle["input_features"]
for leakage_kw in ["is_solved", "disposition", "outcome", "arrest"]:
    assert not any(leakage_kw in col.lower() for col in feature_names_in), f"Leakage found: {leakage_kw}"
print("Validation 2: Zero target leakage detected in feature selection.")

# 3. Cumulative Explained Variance Validity
cum_var = exp_df["cumulative_explained_variance"].values
assert np.all(np.diff(cum_var) >= 0), "Cumulative variance must be monotonically increasing."
assert cum_var[-1] <= 1.0001, "Cumulative variance cannot exceed 1.0."
print("Validation 3: Cumulative explained variance is valid and monotonically increasing.")

# 4. District Aggregations & Rate Denominators
ds = pd.read_csv(OUTPUTS_DIR / "tamil_nadu" / "district_summary.csv")
assert "TOTAL DISTRICT(S)" not in ds["district"].values, "Total row must be filtered."
for num_col in ds.select_dtypes(include=[np.number]).columns:
    assert (ds[num_col].dropna() >= 0).all(), f"Negative values in {num_col}"
print("Validation 4: District summaries exclude aggregate rows and have non-negative crime counts.")

# 5. Raw Data Immutability
for rf in RAW_DATA_DIR.glob("*.csv"):
    assert rf.stat().st_mtime < model_path.stat().st_mtime, f"Raw file modified! {rf.name}"
print("Validation 5: Raw datasets in data/raw/ remain strictly unmodified.")

print("\nALL DAY 3 VALIDATION CHECKS PASSED!")